# 训练数据准备

从现有的文本数据集中抽取适合朗读的条目，用语音合成（TTS）把它们转为音频，再用语音识别（ASR）把音频转回文字，筛选内容一致的音频—文本对，用于训练音频 projector。数据覆盖 18 种语言：中文、英语、日语、韩语、德语、法语、俄语、葡萄牙语、西班牙语、意大利语、印尼语、马来语、泰语、越南语、菲律宾语、阿拉伯语、土耳其语和波斯语。

整个流程分五步，每一步只读取上一步写出的文件，可以单独重跑：

```text
文本数据集 ─①抽取─▶ 原文 ─②改写─▶ 朗读稿 ─④TTS 合成─▶ 音频 ─⑤ASR 识别并比对─▶ 训练清单
                                    ③准备音色 ──┘
```

代码在仓库的 `pipeline/` 目录下，按步骤分成子目录：`text/` 抽取原文，`tn/` 生成朗读稿，`voices/` 准备音色，`tts/` 和 `asr/` 是各个模型的调用封装，`compare/` 比对朗读稿与识别结果，`runner/` 把合成和核对串成流水线。不同的 TTS、ASR 模型依赖的包版本互相冲突，所以每个模型装在自己的虚拟环境里，流水线以子进程调用它们；环境的建立方法见 `pipeline/ENVIRONMENTS.md`。完整规模的数据用命令行脚本生成，命令顺序见 `pipeline/README.md`。本 notebook 用小样本演示每一步的操作，再读取完整规模运行的结果，说明数据的情况。

## 0. 准备路径

流水线用到两个目录。工作目录存放各模型的虚拟环境、GPU 锁文件和日志，用环境变量 `WORK_DIR` 指定，没有设置时取与仓库目录并列的 `qwen3.5-asr-work/`。数据集目录用环境变量 `DATASET` 指定：抽取的文本放在 `$DATASET/text/qwen3.5-asr/` 下，合成的音频放在 `$DATASET/audio/qwen3.5-asr/` 下。本 notebook 的演示文件放在工作目录的 `tmp/tutorial/` 中。

各步骤之间用 JSONL 文件交换数据：每行是一个 JSON 对象，代表一条文本或一个样本。`read_jsonl` 和 `write_jsonl` 读写这种文件。`languages` 列出十八种语言的代码。

In [1]:
import json
import os
import re
import subprocess
from collections import Counter
from pathlib import Path

repo_dir = Path.cwd()
work_dir = Path(os.environ.get("WORK_DIR", repo_dir.parent / "qwen3.5-asr-work"))
dataset_dir = Path(os.environ["DATASET"])
text_dir = dataset_dir / "text" / "qwen3.5-asr"
audio_dir = dataset_dir / "audio" / "qwen3.5-asr"
demo_dir = work_dir / "tmp" / "tutorial"
demo_dir.mkdir(parents = True, exist_ok = True)

languages = [
    "zh", "en", "ja", "ko", "de", "fr", "ru", "pt", "es", "it",
    "id", "ms", "th", "vi", "fil", "ar", "tr", "fa",
]


def read_jsonl(path):
    with open(file = path, mode = "r", encoding = "utf-8") as handle:
        return [json.loads(s = line) for line in handle if line.strip()]


def write_jsonl(path, rows):
    with open(file = path, mode = "w", encoding = "utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(obj = row, ensure_ascii = False) + "\n")

## 1. 从现有数据集抽取原文

适合朗读的原文只使用目标语言，由完整、连贯的句子组成，不含代码、数学公式、标题、列表等无法自然读出的内容。抽取出的每条文本叫一条「原文条目」，保存语言、来源数据集、许可和原文。

### 1.1 文本来源

文本来自两类数据集。短句来源的句子完整、干净，内容偏日常；网页和新闻来源提供更长、更书面的句子。每种语言抽约 4 万条，按来源配额混合。

| 来源 | 内容 | 许可 |
| --- | --- | --- |
| Tatoeba | 多语种例句 | CC BY 2.0 FR |
| Common Voice 句库 | 为众包录音准备的短句 | CC0 |
| singletongue/wikipedia-paragraphs | 维基百科段落 | CC BY-SA 4.0 |
| HPLT 3.0 | 网页文本 | 整理工作为 CC0，原文版权归作者 |
| FineWeb2-HQ-Plus、fineweb-edu | 筛选过的网页文本 | ODC-By |
| Wikinews | 新闻 | 多数为 CC BY 4.0 |

各来源的调查和许可说明见仓库根目录的 `datasets.md`。每条原文记录来源数据集（`source`）和许可（`source_license`），以后需要按许可筛选时可以追溯。

### 1.2 过滤不能直接朗读的文本

每条候选文本先经过 `pipeline/text/filters.py` 的 `check_text` 检查。这个函数只看文本本身和语言代码：能直接朗读时返回 `None`，否则返回丢弃原因。主要的丢弃原因有：含网址、邮箱、代码或标记语言（`url`、`email`、`code`、`markup`），不以句末标点结尾（`no_terminal`），像列表或标题（`list_like`），文字不属于该语言的书写系统（`script`），太短或太长（`too_short`、`too_long`），符号或长串数字过多（`symbols`、`long_number`）。下面用几句例子演示。

In [2]:
from pipeline.text.filters import check_text

examples = [
    ("zh", "今天天气很好，我们去公园散步吧。"),
    ("en", "Visit https://example.com for more details."),
    ("en", "def area(r): return 3.14 * r * r"),
    ("en", "the cat sat on the mat"),
    ("de", "Die Sitzung beginnt um 9 Uhr."),
]
for lang, text in examples:
    print(f"{lang}  {str(check_text(text = text, lang = lang)):12}  {text}")

zh  None          今天天气很好，我们去公园散步吧。
en  url           Visit https://example.com for more details.
en  code          def area(r): return 3.14 * r * r
en  no_terminal   the cat sat on the mat
de  None          Die Sitzung beginnt um 9 Uhr.


中文和德语两句通过检查，德语句子里的数字「9」留给下一步改写成读法。英语三句分别因为网址、代码和缺少句末标点被丢弃。

通过检查的文本再用语言识别模型 GlotLID 确认语言，去掉重复的句子，并去掉与 FLEURS 相同的句子。FLEURS 是后面用来评测模型的真人朗读录音集，训练文本里如果出现评测用的句子，评测结果会偏高。

### 1.3 按估计时长分档

训练数据需要长短不同的音频。合成之前还没有音频，只能用字符数估计朗读时长：每种语言有一个估计语速（字符/秒），取自 FLEURS 录音的转写字符数除以录音时长的中位数，保存在 `pipeline/text/langs.py` 的 `CHARS_PER_SECOND` 中。估计时长 1–5 秒的条目为短（`short`），5–15 秒为中（`medium`），15–30 秒为长（`long`），超出这个范围的不用。

In [3]:
from pipeline.text.langs import CHARS_PER_SECOND
from pipeline.text.lengths import estimate_duration, length_bin

for lang, text in [
    ("zh", "今天天气很好，我们去公园散步吧。"),
    ("de", "Die Sitzung beginnt um 9 Uhr."),
]:
    seconds = estimate_duration(text = text, lang = lang)
    print(f"{lang}：{len(text)} 个字符，语速 {CHARS_PER_SECOND[lang]} 字符/秒，估计 {seconds:.1f} 秒，档位 {length_bin(duration_s = seconds)}")

zh：16 个字符，语速 3.69 字符/秒，估计 4.3 秒，档位 short
de：29 个字符，语速 12.33 字符/秒，估计 2.4 秒，档位 short


### 1.4 抽取结果

抽取的命令见 `pipeline/README.md` 第 2 节：`pipeline.text.extract` 处理短句来源，`pipeline.text.extract_web` 处理网页和新闻来源，`pipeline.text.merge` 按来源配额合并成每种语言的最终条目，写在 `$DATASET/text/qwen3.5-asr/source-text/final/<语言>.jsonl`。下面统计每种语言的条目数，再看中文条目的来源构成、长度分档和一条条目的字段。

In [4]:
final_dir = text_dir / "source-text" / "final"
counts = {lang: len(read_jsonl(path = final_dir / f"{lang}.jsonl")) for lang in languages}
print("每种语言的条目数：", counts)

zh_entries = read_jsonl(path = final_dir / "zh.jsonl")
print("中文来源：", Counter(row["source"] for row in zh_entries).most_common())
print("中文长度分档：", Counter(row["length_bin"] for row in zh_entries))
fields = ["entry_id", "lang", "source", "source_license", "original", "n_chars", "length_bin", "est_duration_s"]
print(json.dumps(obj = {key: zh_entries[0][key] for key in fields}, ensure_ascii = False, indent = 1))

每种语言的条目数： {'zh': 40000, 'en': 40000, 'ja': 40000, 'ko': 40000, 'de': 40000, 'fr': 40000, 'ru': 40000, 'pt': 40000, 'es': 40000, 'it': 40000, 'id': 42608, 'ms': 40000, 'th': 45144, 'vi': 40216, 'fil': 40020, 'ar': 40000, 'tr': 41406, 'fa': 41538}
中文来源： [('singletongue/wikipedia-paragraphs', 15049), ('Tatoeba', 7587), ('Common Voice 句库', 7364), ('HPLT 3.0', 6000), ('Wikinews', 2000), ('epfml/FineWeb2-HQ-Plus', 2000)]
中文长度分档： Counter({'medium': 18000, 'short': 12000, 'long': 10000})
{
 "entry_id": "zh-tatoeba-000001",
 "lang": "zh",
 "source": "Tatoeba",
 "source_license": "CC-BY-2.0-FR",
 "original": "“如果不努力工作就不会拥有成功的人生”，这种说法得到广泛的认可。",
 "n_chars": 32,
 "length_bin": "medium",
 "est_duration_s": 8.7
}


## 2. 改写成朗读稿

原文再统一两处书写，得到交给 TTS 的朗读稿：中文统一转为简体；数字、单位和缩写改写成实际读法，例如「3.5公里」写成「三点五公里」。直接把原文交给 TTS，「2025」可能被念成「两千零二十五」或「二零二五」，「%」可能被漏掉；先写成确定的读法，TTS 照着念，ASR 核对时也有一致的依据。

原文和朗读稿分开保存，因为训练时用哪种写法作为目标文本，模型就学会输出哪种写法：用朗读稿，模型听到「3.5公里」会输出「三点五公里」；用原文，则输出「3.5公里」。两份都保存，到训练阶段再选择。

改写由 `pipeline/tn` 完成，每种语言有自己的规则文件（`pipeline/tn/langs/<语言>.py`），处理数字、日期、时间、百分号、货币、单位、序数和常见缩写；俄语、阿拉伯语等需要按上下文选择数词的格和性。每条条目改写后得到三种状态之一：`unchanged` 表示没有需要改写的内容，朗读稿与原文相同；`normalized` 表示已经改写；`dropped` 表示含有无法确定读法的内容（例如比分、经文编号、残缺的片段），这条不参加合成。

下面从五种语言的最终条目中各取两条含数字的短句，用 `envs/tn` 环境运行 `pipeline.tn.run`，比较原文和朗读稿。

In [5]:
picked = []
for lang in ["zh", "en", "ja", "ru", "ar"]:
    rows = read_jsonl(path = final_dir / f"{lang}.jsonl")
    with_digits = [row for row in rows if row["length_bin"] == "short" and re.search(pattern = r"\d", string = row["original"])]
    picked += with_digits[:2]
tn_input = demo_dir / "tn_demo_in.jsonl"
tn_output = demo_dir / "tn_demo_out.jsonl"
write_jsonl(path = tn_input, rows = picked)

subprocess.run(
    args = [str(work_dir / "envs" / "tn" / "bin" / "python"), "-m", "pipeline.tn.run", "--input", str(tn_input), "--output", str(tn_output)],
    check = True,
    capture_output = True,
)
for row in read_jsonl(path = tn_output):
    print(f"{row['lang']}  {row['tn_status']}")
    print(f"    原文：{row['original']}")
    print(f"    朗读稿：{row['script'] if row['script'] else '（丢弃：' + str(row['tn_note']) + '）'}")

zh  normalized
    原文：足總杯8強被阿仙奴淘汰。
    朗读稿：足总杯八强被阿仙奴淘汰。
zh  normalized
    原文：還有 4 分鐘就下課了，請準備下課！
    朗读稿：还有四分钟就下课了，请准备下课！
en  normalized
    原文：95.3% spoke English and 2.6% Spanish as their first language.
    朗读稿：Ninety-five point three percent spoke English and two point six percent Spanish as their first language.
en  normalized
    原文：In captivity, a giraffe may live up to 40 years old.
    朗读稿：In captivity, a giraffe may live up to forty years old.
ja  normalized
    原文：７時にきっかりに必ずここにきなさい。
    朗读稿：七時にきっかりに必ずここにきなさい。
ja  normalized
    原文：第2回手塚治虫文化賞マンガ大賞受賞作品。
    朗读稿：第二回手塚治虫文化賞マンガ大賞受賞作品。
ru  dropped
    原文：13:19), без которых евреи не могли покинуть Египет (в Быт.
    朗读稿：（丢弃：冒号与数字相连「13:19」）
ru  normalized
    原文：В октябре 2001 года был переизбран на эту должность.
    朗读稿：В октябре две тысячи первого года был переизбран на эту должность.
ar  dropped
    原文：2 مباراتي الذهاب والإياب لعبا في البهاما.
    朗读稿：（丢弃：阿拉伯语数词与后面名词的性、数、格一致无法确定「2」）
ar  dropped
    原文：فاز تشيلسي بالنهاية 3–1.
 

中文两条转成了简体，「8強」「4 分鐘」改写成「八强」「四分钟」。英语的百分数、日语的全角数字「７」都改写成了读法。俄语「2001 года」改写成「две тысячи первого года」：俄语的年份用序数词，还要跟着后面的「года」变成属格，规则文件要按上下文选择词形。

被丢弃的三条各有原因。俄语那条以「13:19)」开头，是从经文引用中间截断的片段，冒号连着的数字无法确定读法。阿拉伯语的数词要与后面名词的性、数、格一致，第一条里「2」的读法无法可靠判断；第二条的「3–1」可能是比分、范围或编号，读法各不相同。把这类条目丢弃，比猜一个可能错误的读法更稳妥。

`pipeline.tn.run` 在每个输出文件旁边写一份 `<语言>.summary.json`，记录三种状态的条数。下面读取完整规模运行的统计。

In [6]:
print("语言  条目数  不需改写  已改写  丢弃")
for lang in languages:
    summary = json.loads(s = (text_dir / "scripts-v1" / f"{lang}.summary.json").read_text(encoding = "utf-8"))
    item = summary["langs"][lang]
    ratio = item["ratio"]
    print(f"{lang:4}  {item['n']:6}  {ratio['unchanged']:7.1%}  {ratio['normalized']:6.1%}  {ratio['dropped']:5.1%}")

语言  条目数  不需改写  已改写  丢弃
zh     40000    83.2%   14.0%   2.8%
en     40000    76.9%   19.1%   4.0%
ja     40000    79.2%   17.9%   2.9%
ko     40000    77.0%   16.3%   6.7%
de     40000    80.1%   14.9%   5.0%
fr     40000    75.5%   19.7%   4.8%
ru     40000    79.9%   11.7%   8.3%
pt     40000    81.3%   14.2%   4.4%
es     40000    79.8%   15.6%   4.6%
it     40000    77.1%   18.2%   4.7%
id     42608    88.6%    7.5%   3.9%
ms     40000    78.0%   15.0%   7.1%
th     45144    86.7%    9.1%   4.2%
vi     40216    82.2%   13.3%   4.5%
fil    40020    93.9%    3.7%   2.4%
ar     40000    75.5%   12.7%  11.7%
tr     41406    91.2%    6.3%   2.5%
fa     41538    92.9%    6.4%   0.8%


大多数语言约 75%–94% 的条目不需要改写，约 4%–20% 改写成了读法，丢弃的比例在 1%–12% 之间。阿拉伯语丢弃得最多（11.7%），主要因为数词与名词的一致关系难以确定；俄语（8.3%）也受变格影响。改写结果写在 `$DATASET/text/qwen3.5-asr/scripts-v1/<语言>.jsonl`，完整规模的运行命令是 `pipeline/tn/run_all.sh`。

## 3. 准备音色

TTS 合成时要指定说话人的声音，也就是音色。本项目用两种方式指定音色：

- 内置音色（`builtin`）：Qwen3-TTS、Supertonic-3、VieNeu 自带若干固定的说话人，按名字选用。
- 声音克隆（`clone`）：OmniVoice、VoxCPM2、IndexTTS 没有固定的说话人。给模型一段参考录音和录音对应的文字，模型模仿录音里的声音来朗读新句子。

克隆用的参考录音由 `pipeline/voices/build_voices.py` 生成：用三个模型的每个内置音色朗读一段 8–12 秒的句子，经 ASR 核对无误后保存。所以克隆音色的参考录音本身也是合成的。音色库写在 `<工作目录>/data/T07/voices.jsonl`，每行一个音色。

每个模型留出几个音色（`holdout` 为 true），只用来合成测试集，训练数据从不使用。测试时用这些音色，可以检验模型对没听过的声音是否同样有效。

In [7]:
voices = read_jsonl(path = work_dir / "data" / "T07" / "voices.jsonl")
print("音色数：", len(voices))
print("各模型的音色数：", Counter(voice["source_model"] for voice in voices))
print("留出音色：", [voice["voice_id"] for voice in voices if voice["holdout"]])
fields = ["voice_id", "source_model", "builtin_name", "gender", "ref_lang", "ref_text", "holdout"]
print(json.dumps(obj = {key: voices[0][key] for key in fields}, ensure_ascii = False, indent = 1))

音色数： 42
各模型的音色数： Counter({'vieneu-v3-turbo': 23, 'supertonic-3': 10, 'qwen3-tts': 9})
留出音色： ['qwen3tts-Eric', 'supertonic3-F5', 'supertonic3-M5', 'vieneu-Minh_Triet', 'vieneu-Thuy_Dung', 'vieneu-Kim_Thanh']
{
 "voice_id": "qwen3tts-Vivian",
 "source_model": "qwen3-tts",
 "builtin_name": "Vivian",
 "gender": "female",
 "ref_lang": "zh",
 "ref_text": "艾尔斯在里奇蒙大学数位史学实验室的工作专注在显示出美国历史中的趋势与模式。",
 "holdout": false
}


## 4. 合成音频并核对文字

用 TTS 把朗读稿转为对应语言的音频，再用 ASR 识别音频中的文字。对朗读稿与识别结果采用相同的文本规范化规则，统一约定的书写差异；只有规范化后完全一致的样本才保留为音频—文本对。

### 4.1 每种语言用哪些模型

没有一个开源 TTS 模型能把 18 种语言都念好，ASR 也是一样。`pipeline/runner/config_v1.yaml` 为每种语言指定：

- `tts`：可用的 TTS 模型和抽取权重。每条朗读稿按权重随机抽一个 TTS 模型，再从这个模型可用的音色中随机抽一个。模型有支持该语言的内置音色时用内置音色，否则用克隆。权重按试验中各模型在该语言上的核对通过率和合成速度确定。
- `asr`：主 ASR，识别结果必须与朗读稿一致。
- `cross_asr`：交叉 ASR，主 ASR 一致之后再用另一个模型识别一次，也必须一致。没有第二个可靠 ASR 的语言不设交叉 ASR。

In [8]:
import yaml

config = yaml.safe_load(stream = (repo_dir / "pipeline" / "runner" / "config_v1.yaml").read_text(encoding = "utf-8"))
print("语言  主 ASR          交叉 ASR        TTS 模型与权重")
for lang in languages:
    item = config["langs"][lang]
    print(f"{lang:4}  {item['asr']:15} {str(item.get('cross_asr')):15} {item['tts']}")

语言  主 ASR          交叉 ASR        TTS 模型与权重
zh    qwen3_asr       whisper         {'qwen3_tts': 3, 'omnivoice': 1}
en    qwen3_asr       whisper         {'supertonic3': 4, 'omnivoice': 2, 'qwen3_tts': 2}
ja    haas_ja         qwen3_asr       {'qwen3_tts': 3, 'supertonic3': 3, 'omnivoice': 1}
ko    qwen3_asr       whisper         {'supertonic3': 4, 'qwen3_tts': 3, 'omnivoice': 1}
de    qwen3_asr       whisper         {'qwen3_tts': 3, 'supertonic3': 3, 'omnivoice': 2}
fr    qwen3_asr       whisper         {'qwen3_tts': 2, 'supertonic3': 2, 'omnivoice': 2}
ru    gigaam          qwen3_asr       {'supertonic3': 4, 'voxcpm2': 1, 'omnivoice': 1}
pt    qwen3_asr       whisper         {'supertonic3': 3, 'qwen3_tts': 3, 'voxcpm2': 1, 'omnivoice': 1}
es    qwen3_asr       whisper         {'qwen3_tts': 4, 'omnivoice': 1, 'supertonic3': 1, 'voxcpm2': 1}
it    qwen3_asr       whisper         {'qwen3_tts': 4, 'supertonic3': 2, 'omnivoice': 1}
id    qwen3_asr       whisper         {'supertonic3': 4, 'v

### 4.2 核对规则

一个样本要依次通过三项检查才算通过：

1. 时长检查：音频时长除以朗读稿的字符数，要落在该语言的范围内（配置中的 `sec_per_char`）。TTS 偶尔会提前停止或在句尾不停地念下去，这一步直接筛掉这类音频。
2. 主 ASR 比对：用 `pipeline.compare` 比较朗读稿和识别结果。两边先做同一套规范化：Unicode 统一、大小写统一、删除标点、各种数字字形统一，中文转简体，日语平假名转片假名。然后把数词统一成数值，「二十」和「20」视为相同；日语按写法不一致时再按读音比较一次。识别结果里写成阿拉伯数字的数，还必须能在原文中找到，用来发现朗读稿把数写错、TTS 照着念、ASR 照着写的情况。
3. 交叉 ASR 比对：同样的规则，换一个 ASR 模型。单个 ASR 可能凭语言习惯把念错的地方「纠正」回去，两个独立模型都一致，误收的可能就小得多。

### 4.3 小规模演示

下面从朗读稿中为中文、英语、日语、西班牙语各抽 3 条没用过的条目，用训练音色走一遍完整流程。`pipeline.runner.rounds test` 负责抽取：每种语言随机取 3 条，排除已经用于训练、测试和音色参考的条目。`pipeline/runner/run_dir.sh` 依次执行规划任务、合成、时长检查、主 ASR 识别与比对、交叉 ASR 识别与比对、写出清单，每一步的结果都写在输出目录里。合成时 Supertonic-3 在 CPU 上运行，其他模型在 GPU 上运行，两者同时进行。几个模型要依次加载，整个过程约 4 分钟。

In [9]:
demo_entries = demo_dir / "demo_entries.jsonl"
demo_out = demo_dir / "demo_synth"
used_files = sorted((text_dir / "rounds").glob(pattern = "*.jsonl")) + [
    text_dir / "smoke" / "entries.jsonl",
    text_dir / "pilot" / "entries-builtin.jsonl",
    text_dir / "pilot" / "entries-clone.jsonl",
]
runner_python = str(work_dir / "envs" / "runner" / "bin" / "python")
subprocess.run(
    args = [
        runner_python, "-m", "pipeline.runner.rounds", "test",
        "--name", "tutorial",
        "--per-lang", "3",
        "--langs", "zh", "en", "ja", "es",
        "--scripts-dir", str(text_dir / "scripts-v1"),
        "--voices", str(work_dir / "data" / "T07" / "voices.jsonl"),
        "--exclude", *[str(path) for path in used_files],
        "--output", str(demo_entries),
    ],
    check = True,
    capture_output = True,
)
result = subprocess.run(
    args = ["pipeline/runner/run_dir.sh", "tutorial", "train", str(demo_entries), str(demo_out)],
    check = True,
    capture_output = True,
    text = True,
)
print(result.stdout)

=== 2026-11-08 20:05:03 开始 plan
=== 2026-11-08 20:05:03 完成 plan，用时 0 秒
=== 2026-11-08 20:05:03 开始 tts（GPU 与 CPU 并行）
=== 2026-11-08 20:06:51 完成 tts，用时 108 秒
=== 2026-11-08 20:06:51 开始 length
=== 2026-11-08 20:06:51 完成 length，用时 0 秒
=== 2026-11-08 20:06:51 开始 asr-primary
=== 2026-11-08 20:07:58 完成 asr-primary，用时 67 秒
=== 2026-11-08 20:07:58 开始 compare-1
=== 2026-11-08 20:07:58 完成 compare-1，用时 0 秒
=== 2026-11-08 20:07:58 开始 asr-cross
=== 2026-11-08 20:08:36 完成 asr-cross，用时 38 秒
=== 2026-11-08 20:08:36 开始 compare-2
=== 2026-11-08 20:08:36 完成 compare-2，用时 0 秒
=== 2026-11-08 20:08:36 开始 manifest
=== 2026-11-08 20:08:36 完成 manifest，用时 0 秒
=== 全部完成



输出目录里，`tasks.jsonl` 记录每个样本用哪个 TTS 模型和音色，`samples.jsonl` 记录每个样本是否通过及原因，`asr_results.jsonl` 记录每个 ASR 的识别结果、是否一致（`match`）和差异说明（`compare_detail`）。下面把三者合在一起逐条列出。

In [10]:
tasks = {row["sample_id"]: row for row in read_jsonl(path = demo_out / "tasks.jsonl")}
asr_rows = {}
for row in read_jsonl(path = demo_out / "asr_results.jsonl"):
    asr_rows.setdefault(row["sample_id"], []).append(row)
samples = read_jsonl(path = demo_out / "samples.jsonl")
for sample in samples:
    task = tasks[sample["sample_id"]]
    print(f"{sample['sample_id']}  {task['tts_model']} {task['mode']} {task['voice_id']}  → {sample['status']} {sample.get('reason') or ''}")
    print(f"    朗读稿：{task['text']}")
    for asr in sorted(asr_rows.get(sample["sample_id"], []), key = lambda row: row["asr_role"] != "primary"):
        verdict = "一致" if asr["match"] else "不一致：" + asr["compare_detail"]
        print(f"    {asr['asr_role']:7} {asr['asr_model']:16} {verdict}")
        print(f"            {asr['hypothesis']}")

train-en-hplt-000007  supertonic3 builtin supertonic3-F1  → pass 
    朗读稿：In the workplace we need to adapt to meet this expectation along with the idea that our users not only want to access data when they want but from whatever device they choose.
    primary qwen3-asr-1.7b   一致
            In the workplace, we need to adapt to meet this expectation, along with the idea that our users not only want to access data when they want, but from whatever device they choose.
    cross   whisper-large-v3 一致
            In the workplace we need to adapt to meet this expectation along with the idea that our users not only want to access data when they want, but from whatever device they choose.
train-en-tatoeba-003695  qwen3_tts builtin qwen3tts-Vivian  → pass 
    朗读稿：His wife pushed him away, saying that her migraine had taken hold again.
    primary qwen3-asr-1.7b   一致
            His wife pushed him away, saying that her migraine had taken hold again.
    cross   whisper-large-v3 一致
        

12 条中 9 条通过。没通过的 3 条正好代表三种常见情况：

- 中文那条，Qwen3-TTS 念得没有问题，主 ASR 把「统统」写成了同音的「通通」。比对只合并约定的书写差异，同音不同字算不一致，这条被筛掉。核对标准偏严，会损失一些其实正确的音频，换来的是留下的文字与音频严格对应。
- 日语那条，OmniVoice 克隆的音频有几处念得含糊，主 ASR 把「航海」识别成「後悔」，把「ガレオン船」识别成「ガレ温泉」。这是合成音频本身的错误，应该筛掉。
- 西班牙语那条，VoxCPM2 在句尾多出一段不在朗读稿里的声音，主 ASR 把它写成「Exensai」。TTS 多念或漏念的内容会被 ASR 如实写出来，比对时就能发现。

通过的样本里也能看到规范化的作用：ASR 自己加上的逗号和句号、Haas-ja 把「AI」写成小写的「ai」都不影响判定；日语「漏えい」和「漏洩」写法不同，按读音比较时相同，判为一致。

下面播放一条通过的英语音频和那条没通过的西班牙语音频，可以听到句尾多出的声音。

In [11]:
from IPython.display import Audio, display

passed_id = next(sample["sample_id"] for sample in samples if sample["status"] == "pass")
failed_id = next(sample["sample_id"] for sample in samples if sample["status"] == "fail")
for label, sample_id in (("通过", passed_id), ("没通过", failed_id)):
    print(label, sample_id, tasks[sample_id]["text"])
    display(Audio(filename = str(demo_out / "audio" / f"{sample_id}.flac")))

通过 train-en-hplt-000007 In the workplace we need to adapt to meet this expectation along with the idea that our users not only want to access data when they want but from whatever device they choose.


没通过 train-es-wikipara-002514 En mil ochocientos ochenta y seis, fue nombrado cónsul general en Dinamarca .


### 4.4 第一轮的完整结果

完整规模的数据按轮生成。第一轮先用 `pipeline.runner.rounds test` 为每种语言抽 60 条测试条目，用留出音色合成测试集；再用 `pipeline.runner.rounds train` 按「每种语言约 5 小时通过的音频」抽取训练条目：根据试验中各 TTS 模型的通过率估计每种语言需要合成多少条，短、中、长三档按 30%、45%、25% 抽取。`pipeline/runner/run_round.sh r1` 按语言依次运行，每种语言一个输出目录，在 `$DATASET/audio/qwen3.5-asr/v1/train/<语言>/`。下面读取每种语言的统计和清单。

In [12]:
round_dir = audio_dir / "v1" / "train"
total_tasks, total_pass, total_seconds = 0, 0, 0.0
print("语言  合成条数  通过条数  通过率  通过音频（小时）")
for lang in languages:
    stats = json.loads(s = (round_dir / lang / "stats.json").read_text(encoding = "utf-8"))
    seconds = sum(row["duration_s"] for row in read_jsonl(path = round_dir / lang / "manifest.jsonl"))
    total_tasks += stats["total"]
    total_pass += stats["pass"]
    total_seconds += seconds
    print(f"{lang:4}  {stats['total']:8}  {stats['pass']:8}  {stats['rate']:6.1%}  {seconds / 3600:8.2f}")
print(f"合计  {total_tasks:8}  {total_pass:8}  {total_pass / total_tasks:6.1%}  {total_seconds / 3600:8.2f}")

语言  合成条数  通过条数  通过率  通过音频（小时）
zh        7043      2432   34.5%      4.54
en        3622      2443   67.5%      5.24
ja        7742      3325   43.0%      4.85
ko        4400      2365   53.8%      3.98
de        4528      2472   54.6%      4.88
fr        6286      2303   36.6%      4.50
ru        5174      2409   46.6%      4.59
pt        4579      2084   45.5%      3.81
es        3893      2166   55.6%      4.77
it        4339      2190   50.5%      3.78
id        4108      2490   60.6%      5.08
ms       10000      3337   33.4%      8.05
th        7625      2726   35.8%      4.42
vi        6339      2851   45.0%      4.38
fil       7862      2207   28.1%      2.72
ar        4277      2166   50.6%      5.08
tr        3630      2269   62.5%      3.91
fa       10000      1543   15.4%      2.61
合计    105447     43778   41.5%     81.17


In [13]:
by_model = {}
by_mode = {}
reasons = Counter()
for lang in languages:
    lang_tasks = {row["sample_id"]: row for row in read_jsonl(path = round_dir / lang / "tasks.jsonl")}
    for sample in read_jsonl(path = round_dir / lang / "samples.jsonl"):
        task = lang_tasks[sample["sample_id"]]
        passed = sample["status"] == "pass"
        for table, key in ((by_model, task["tts_model"]), (by_mode, task["mode"])):
            table.setdefault(key, [0, 0])
            table[key][0] += passed
            table[key][1] += 1
        if not passed:
            reasons[sample["reason"]] += 1
for name, table in (("TTS 模型", by_model), ("合成方式", by_mode)):
    print(f"按{name}：")
    for key, (passed, total) in sorted(table.items(), key = lambda item: -item[1][1]):
        print(f"    {key:12} {total:6} 条，通过率 {passed / total:.1%}")
print("没通过的原因：", reasons.most_common())

按TTS 模型：
    omnivoice     39309 条，通过率 28.0%
    supertonic3   26172 条，通过率 59.0%
    qwen3_tts     21110 条，通过率 47.0%
    voxcpm2       15434 条，通过率 37.3%
    vieneu         2408 条，通过率 41.6%
    indextts25     1014 条，通过率 65.2%
按合成方式：
    clone         55757 条，通过率 31.3%
    builtin       49690 条，通过率 53.0%
没通过的原因： [('mismatch', 53519), ('cross_mismatch', 7842), ('length_too_long', 158), ('asr_empty', 76), ('tts_failed', 69), ('length_too_short', 5)]


第一轮合成了 105,447 条，通过 43,778 条，共 81.2 小时。各语言通过的时长在 2.6–8.1 小时之间。按估计通过率抽样后，大多数语言接近 5 小时的目标；马来语句子偏长，通过条数不多，时长却最多；波斯语只有 OmniVoice 一个 TTS 可用，通过率只有 15.4%，菲律宾语的句子偏短，这两种语言通过的音频都不到 3 小时。

按 TTS 模型看，Supertonic-3 的通过率最高（59.0%；IndexTTS 只合成了 1,014 条，不作比较），OmniVoice 最低（28.0%），但 OmniVoice 支持的语言最多，承担了最多的合成量。内置音色的通过率（53.0%）明显高于克隆（31.3%）。没通过的样本绝大多数是主 ASR 不一致（53,519 条），其次是交叉 ASR 不一致（7,842 条）；时长检查、合成失败和识别结果为空加起来约 300 条。

这一轮约用 20 小时机器时间。这份数据训练出的模型效果见下一个教程和 `results/v1-zh.md`。

## 5. 训练清单

每个输出目录的 `manifest.jsonl` 是核对通过的样本，一行一个音频—文本对，训练只读这个文件。它合并了前面各步的信息：音频路径（相对输出目录）和时长、语言、原文、朗读稿、来源与许可、TTS 模型、音色、是否留出音色、是否跨语言克隆、ASR 模型等。下面看演示目录中的一行。

In [14]:
row = read_jsonl(path = demo_out / "manifest.jsonl")[0]
fields = ["sample_id", "audio_path", "duration_s", "lang", "original", "script", "source", "tts_model", "voice_id", "mode", "holdout", "cross_lingual", "asr_model", "cross_asr_model"]
print(json.dumps(obj = {key: row.get(key) for key in fields}, ensure_ascii = False, indent = 1))

{
 "sample_id": "train-en-hplt-000007",
 "audio_path": "audio/train-en-hplt-000007.flac",
 "duration_s": 10.747,
 "lang": "en",
 "original": "In the workplace we need to adapt to meet this expectation along with the idea that our users not only want to access data when they want but from whatever device they choose.",
 "script": "In the workplace we need to adapt to meet this expectation along with the idea that our users not only want to access data when they want but from whatever device they choose.",
 "source": "HPLT 3.0",
 "tts_model": "supertonic3",
 "voice_id": "supertonic3-F1",
 "mode": "builtin",
 "holdout": false,
 "cross_lingual": true,
 "asr_model": "qwen3-asr-1.7b",
 "cross_asr_model": "whisper-large-v3"
}


`cross_lingual` 表示参考录音的语言与要念的语言不同。内置音色和克隆参考录音的说话人大多只对应一种语言，例如用英语音色念西班牙语，这时 `cross_lingual` 为 true。

训练前用 `pipeline.runner.merge_manifests` 把 18 种语言的清单合成一份，音频路径改为绝对路径：

```bash
python -m pipeline.runner.merge_manifests $DATASET/audio/qwen3.5-asr/v1/train/manifest-r1.jsonl \
    $DATASET/audio/qwen3.5-asr/v1/train/{zh,en,ja,ko,de,fr,ru,pt,es,it,id,ms,th,vi,fil,ar,tr,fa}
```

第一轮合并后共 43,778 条、81.2 小时。下一个教程用这份清单训练音频 projector。